# TP 4: Transfert de modèles, explicabilité et biais

Dans cette séance, nous verrons comment :

   - utiliser un modèle préentrainé pour l'adapter à une nouvelle tâche (transfert)
   - analyser les prédictions du modèle pour comprendre les résultats/analyser les erreurs
   - chercher les biais éventuels du modèle liés aux données d'entrainement (de la tâche ou du modèle préentrainé)

Nous nous intéresserons encore à la tâche d'analyse de sentiments, sur les données anglaises IMDB.
Il s'agit d'une tâche de classification de séquences de mots.
Nous nous appuierons sur la librairie HuggingFace et les modèles de langue Transformer (i.e. BERT).  
- https://huggingface.co/ : une librairie de NLP open-source qui offre une API très riche pour utiliser tester différentes architectures et différents modèles pour les problèmes classiques de classification, sequence tagging, generation ... N'hésitez pas à parcourir les démos et modèles existants : https://huggingface.co/tasks/text-classification
- Un assez grand nombre de jeux de données est aussi accessible directement via l'API, pour le texte ou l'image notamment cf les jeux de données https://huggingface.co/datasets et la doc pour gérer ces données : https://huggingface.co/docs/datasets/index

Le code ci-dessous vous permet d'installer :    
- le module *transformers*, qui contient les modèles de langue https://pypi.org/project/transformers/
- le module *transformers_interpret* : un outil pour l'explicabilité des modèles (qui fonctionne avec le module précédent) https://pypi.org/project/transformers-interpret/
- la librairie de datasets pour accéder à des jeux de données
- la librairie *evaluate* : utilisée pour évaluer et comparer des modèles https://pypi.org/project/evaluate/

In [1]:
%pip -q install -U transformers[torch]
%pip -q install transformers_interpret
%pip -q install datasets
%pip -q install evaluate
#%pip -q install -U sklearn

zsh:1: no matches found: transformers[torch]
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import transformers
import accelerate
from transformers_interpret import SequenceClassificationExplainer, TokenClassificationExplainer
from datasets import load_dataset
import evaluate
import numpy as np
import sklearn

In [3]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoModelForTokenClassification

In [4]:
import pandas as pds
from tqdm import tqdm

# 1 - Charger un modèle pré-entraîné : DistilBERT

Le code de cette section permet de charger le modèle et le tokenizer associé.
Dans cette session, nous choisissons le modèle DistilBERT, une version plus petite et rapide du modèle transformer BERT.

Plus d'info ici: https://huggingface.co/distilbert-base-uncased.


In [5]:
# Chosing the pre-trained model
# - distilBERT: specific, faster and lighter version of BERT
# - base vs large
# - uncased: ignore upper case
base_model = "distilbert-base-uncased"

## 1.1 Tokenizer

Notez que la librairie HuggingFace définit des *Auto Classes*:    
 - elles permettent d'inférer directement l'architecture requise selon le type de modèle spécifié en argument.

 - Par exemple ici, le tokenizer est spécifique au modèle DistilBERT, plus précisément il est identique à celui de BERT, et hérite beaucoup de méthodes de la classe *PreTrainedTokenizerFast*.

Le tokenizer est en charge de préparer les données d'entrée, et notamment dans le cas de BERT, de découper les tokens en sous-tokens, mais aussi d'assigner des `ids` à chaque sous-token, de permettre le mapping dans un sens et dans l'autre...

- Les *Auto Classes*: https://huggingface.co/docs/transformers/model_doc/auto
- Les Tokenizer dans HuggingFace: https://huggingface.co/docs/transformers/v4.25.1/en/main_classes/tokenizer
- *Bert tokenizer*: https://huggingface.co/docs/transformers/v4.25.1/en/model_doc/bert#transformers.BertTokenizer
- Classe *PreTrainedTokenizerFast*: https://huggingface.co/docs/transformers/v4.25.1/en/main_classes/tokenizer#transformers.PreTrainedTokenizerFast

In [6]:
# Defining the tokenizer using Auto Classes
tokenizer = AutoTokenizer.from_pretrained(base_model)
model = AutoModelForSequenceClassification.from_pretrained(base_model)

/Users/asriel/miniconda3/envs/311_env/lib/python3.11/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at distilbert-base-uncased were not used when initializing DistilBertForSequenceClassification: ['vocab_transform.bias', 'vocab_transform.weight', 'vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_layer_norm.weight']
- This IS expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing DistilBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertFo

### ▶▶ Exercice: Tester le tokenizer

**Utiliser le tokenizer pour :**
- `encoder une phrase (en anglais)`:
  * que se passe-t-il dans le cas de mots un peu longs ?
  * de mots inconnus ?
  * Que répresentent les éléments entre crochets ?
- convertir dans l'autre sens : d'une liste d'ids de tokens en texte

In [7]:
text = "We propose an approach to automatically characterize biases that takes into account structural differences and that is efficient for long texts. "
list_IDS =  tokenizer.encode(text)
print(f'list_IDS {list_IDS } len list_IDS {len(list_IDS)} len text {len(text.split())}') 
print(f'len list_IDS {len(list_IDS)} len text {len(text)}') 
print(f'convert IDS to tokens {tokenizer.convert_ids_to_tokens(list_IDS) }') 
print(f'convert IDS to tokens {tokenizer.decode(list_IDS) }') 

list_IDS [101, 2057, 16599, 2019, 3921, 2000, 8073, 2839, 4697, 13827, 2229, 2008, 3138, 2046, 4070, 8332, 5966, 1998, 2008, 2003, 8114, 2005, 2146, 6981, 1012, 102] len list_IDS 26 len text 21
len list_IDS 26 len text 145
convert IDS to tokens ['[CLS]', 'we', 'propose', 'an', 'approach', 'to', 'automatically', 'character', '##ize', 'bias', '##es', 'that', 'takes', 'into', 'account', 'structural', 'differences', 'and', 'that', 'is', 'efficient', 'for', 'long', 'texts', '.', '[SEP]']
convert IDS to tokens [CLS] we propose an approach to automatically characterize biases that takes into account structural differences and that is efficient for long texts. [SEP]


## 1.2 Le modèle pré-entraîné

Un modèle pré-entraîné type BERT est un modèle de langue construit avec une tâche spécifique, non supervisée, permettant d'apprendre des associations entre les mots, et donc des représentations des mots dépendantes de leur contexte.

Dans le cas de ce modèle, l'apprentissage se fait en masquant un certain nombre de mots que le modèle doit apprendre à retrouver.

On peut tester la capacité de ce modèle à deviner un mot manquant dans une phrase.    
  
Dans HuggingFace, des pipelines permettent d'exécuter certaines tâches comme celle-ci très facilement, cf le code ci-dessous.

https://huggingface.co/docs/transformers/main_classes/pipelines

### ▶▶ **Exercice : fill-mask**  
- Faire tourner le code ci-dessous et vérifier que vous comprenez la sortie affichée.
- Est-ce que les sorties proposées font sens à vos yeux ?

In [8]:
from transformers import pipeline

unmasker = pipeline('fill-mask', model=base_model)
#unmasker("Hello I'm a [MASK] model.")
unmasker("Hello I'm a [MASK] model.")

[{'score': 0.05292833223938942,
  'token': 2535,
  'token_str': 'role',
  'sequence': "hello i'm a role model."},
 {'score': 0.03968574479222298,
  'token': 4827,
  'token_str': 'fashion',
  'sequence': "hello i'm a fashion model."},
 {'score': 0.034743454307317734,
  'token': 2449,
  'token_str': 'business',
  'sequence': "hello i'm a business model."},
 {'score': 0.03462297469377518,
  'token': 2944,
  'token_str': 'model',
  'sequence': "hello i'm a model model."},
 {'score': 0.018145214766263962,
  'token': 11643,
  'token_str': 'modeling',
  'sequence': "hello i'm a modeling model."}]

 - Il associe un score d'association à un contexte au sujet d'une phrase pour un mot manquant

## 1.3 Biais dans les données

Comme identifié dans la littérature, ces modèles contiennent des biais dépendants de leurs données d'entraînement.

- Article e.g. *The Woman Worked as a Babysitter: On Biases in Language Generation*, Sheng et al, EMNLP, 2019  https://aclanthology.org/D19-1339/



### ▶▶ Exercice : Identifier les biais

Ajoutez des tests pour identifier des biais en vous inspirant des exemples ci-dessous : quel type de biais pouvez-vous identifier ?

In [9]:
unmasker("The man with a college degree worked as a [MASK].")

[{'score': 0.08368733525276184,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the man with a college degree worked as a carpenter.'},
 {'score': 0.051656801253557205,
  'token': 7500,
  'token_str': 'farmer',
  'sequence': 'the man with a college degree worked as a farmer.'},
 {'score': 0.04342759773135185,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the man with a college degree worked as a waiter.'},
 {'score': 0.03968983143568039,
  'token': 18968,
  'token_str': 'salesman',
  'sequence': 'the man with a college degree worked as a salesman.'},
 {'score': 0.03496334329247475,
  'token': 15893,
  'token_str': 'mechanic',
  'sequence': 'the man with a college degree worked as a mechanic.'}]

In [10]:
unmasker("The black man with a college degree worked as a [MASK].")

[{'score': 0.07276228070259094,
  'token': 10533,
  'token_str': 'carpenter',
  'sequence': 'the black man with a college degree worked as a carpenter.'},
 {'score': 0.0521608404815197,
  'token': 15610,
  'token_str': 'waiter',
  'sequence': 'the black man with a college degree worked as a waiter.'},
 {'score': 0.04256362468004227,
  'token': 18594,
  'token_str': 'miner',
  'sequence': 'the black man with a college degree worked as a miner.'},
 {'score': 0.03880532830953598,
  'token': 7500,
  'token_str': 'farmer',
  'sequence': 'the black man with a college degree worked as a farmer.'},
 {'score': 0.03137960284948349,
  'token': 14460,
  'token_str': 'policeman',
  'sequence': 'the black man with a college degree worked as a policeman.'}]

In [11]:
unmasker("The women man with a college degree worked as a [MASK].")

[{'score': 0.13275644183158875,
  'token': 6821,
  'token_str': 'nurse',
  'sequence': 'the women man with a college degree worked as a nurse.'},
 {'score': 0.11462681740522385,
  'token': 19215,
  'token_str': 'prostitute',
  'sequence': 'the women man with a college degree worked as a prostitute.'},
 {'score': 0.06336908042430878,
  'token': 10850,
  'token_str': 'maid',
  'sequence': 'the women man with a college degree worked as a maid.'},
 {'score': 0.048533737659454346,
  'token': 22583,
  'token_str': 'housekeeper',
  'sequence': 'the women man with a college degree worked as a housekeeper.'},
 {'score': 0.044458553194999695,
  'token': 3836,
  'token_str': 'teacher',
  'sequence': 'the women man with a college degree worked as a teacher.'}]

# 2 - Transfert: analyse de sentiment

On charge ici l'ensemble de données IMDB qui correspond à de l'analyse de sentiment sur des reviews de films (en anglais).
On va utiliser ces données pour affiner notre modèle pré-entraîné (agnostique) sur la tâche d'analyse de sentiments.

In [12]:
dataset = load_dataset("imdb")

In [13]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

## 2.1 Tokenization des données

Le code ci-dessous permet d'obtenir une version tokenisée du corpus.

### ▶▶ Exercice Tokenisation :

Regardez la doc pour vérifier que vous comprenez la fonction des paramètres utilisées : https://huggingface.co/docs/transformers/v4.25.1/en/main_classes/tokenizer#transformers.PreTrainedTokenizer.

- à quoi sert le padding ?
- à quoi correspond le paramètre 'truncation' ?

Note: pour plus de détails sur la fonction *Map()* https://huggingface.co/docs/datasets/process et aussi https://huggingface.co/docs/datasets/v2.7.1/en/package_reference/main_classes#datasets.Dataset.map

In [14]:
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)


tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

Notez que le tokenizer retourne deux éléments:

- input_ids: the numbers representing the tokens in the text.
- attention_mask: indicates whether a token should be masked or not.

Plus d'info sur les datasets: https://huggingface.co/docs/datasets/use_dataset

In [15]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})

## 2.2 Entraînement / Fine-tuning

Pour l'entraînement du modèle, on définit d'abord
- une configuration via la classe *TrainingArguments*.
- un niveau de 'verbosité'
- une métrique d'évaluation

### ▶▶ Exercice : training arguments

Consultez la doc et ajoutez un paramètre permettant une évaluation automatique du modèle sur l'ensemble d'évaluation.

In [31]:
from transformers import TrainingArguments, Trainer
training_args = TrainingArguments(output_dir="test_trainer",
                                  #no_cuda=True, # mettre à True sur ordi perso sans bon GPU
                                  use_mps_device=True,
                                  per_device_train_batch_size=4,
                                  evaluation_strategy="steps",
                                  eval_steps=100,
                                  num_train_epochs=5,
                                  do_eval=True
                                  )

#training_args = TrainingArguments("test-trainer", evaluation_strategy="epoch")

In [32]:
from transformers.utils import logging

logging.set_verbosity_error()

In [33]:
metric = evaluate.load("accuracy")

In [34]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

### Trainer

Une instance de la classe *Trainer* correspond à une boucle d'entraînement classique, basée sur les éléments définis précédemment.

https://huggingface.co/docs/transformers/main_classes/trainer

On va sélectionner un sous-ensemble des données ici, pour que l'entraînement soit un peu moins long.

In [35]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(100))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(10))

In [36]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

[codecarbon ERROR @ 10:59:23] Error: Another instance of codecarbon is probably running as we find `/var/folders/qp/m4x_nf996xq9pxj7nm2t9ffr0000gn/T/.codecarbon.lock`. Turn off the other instance to be able to run this one or use `allow_multiple_runs` or delete the file. Exiting.


### Lancer l'entraînement

On peut enfin lancer l'entraînement !

In [38]:
import os

trainer.train(  )

[codecarbon WARNING @ 10:59:32] Another instance of codecarbon is already running. Exiting.


{'eval_loss': 3.9092113971710205, 'eval_accuracy': 0.4, 'eval_runtime': 0.5862, 'eval_samples_per_second': 17.059, 'eval_steps_per_second': 3.412, 'epoch': 4.0}


[codecarbon WARNING @ 10:59:59] Another instance of codecarbon is already running. Exiting.


{'train_runtime': 27.1381, 'train_samples_per_second': 18.424, 'train_steps_per_second': 4.606, 'train_loss': 0.027419954299926757, 'epoch': 5.0}


TrainOutput(global_step=125, training_loss=0.027419954299926757, metrics={'train_runtime': 27.1381, 'train_samples_per_second': 18.424, 'train_steps_per_second': 4.606, 'train_loss': 0.027419954299926757, 'epoch': 5.0})

## 2.3 Evaluation

### Evaluation sur un exemple

On teste le modèle sur un exemple de l'ensemble d'évaluation.

In [39]:
import torch
device = torch.device("mps")
model = model.to(device)

ex_eval = small_eval_dataset[1]["text"]
input = tokenizer(ex_eval, return_tensors="pt")
input_ids = input.input_ids.to(device)
print(input_ids.shape)
output = model(input_ids)

print("gold", small_eval_dataset[1]["label"])

print(output)

torch.Size([1, 240])
gold 1
SequenceClassifierOutput(loss=None, logits=tensor([[-4.5096,  4.0276]], device='mps:0', grad_fn=<LinearBackward0>), hidden_states=None, attentions=None)


In [40]:
output["logits"]

tensor([[-4.5096,  4.0276]], device='mps:0', grad_fn=<LinearBackward0>)

In [25]:
pred = np.argmax(output["logits"].cpu().detach().numpy(), axis=-1)
print("Pred", pred)

Pred [1]


In [41]:
print(tokenizer.tokenize(ex_eval))

['this', 'is', 'the', 'latest', 'entry', 'in', 'the', 'long', 'series', 'of', 'films', 'with', 'the', 'french', 'agent', ',', 'o', '.', 's', '.', 's', '.', '117', '(', 'the', 'french', 'answer', 'to', 'james', 'bond', ')', '.', 'the', 'series', 'was', 'launched', 'in', 'the', 'early', '1950', "'", 's', ',', 'and', 'spawned', 'at', 'least', 'eight', 'films', '(', 'none', 'of', 'which', 'was', 'ever', 'released', 'in', 'the', 'u', '.', 's', '.', ')', '.', "'", 'o', '.', 's', '.', 's', '.', '117', ':', 'cairo', ',', 'nest', 'of', 'spies', "'", 'is', 'a', 'bree', '##zy', 'little', 'comedy', 'that', 'should', 'not', '.', '.', '.', 'repeat', 'not', ',', 'be', 'taken', 'too', 'seriously', '.', 'our', 'protagonist', 'finds', 'himself', 'in', 'the', 'middle', 'of', 'a', 'spy', 'chase', 'in', 'egypt', '(', 'with', 'mor', '##ro', '##co', 'doing', 'stand', 'in', 'for', 'egypt', ')', 'to', 'find', 'out', 'about', 'a', 'long', 'lost', 'friend', '.', 'what', 'follows', 'is', 'the', 'standard', 'james

### ▶▶ Exercice : Analyse d'erreurs

 - Affichez les exemples sur lesquels le modèle a fait une erreur de prédiction.     
 - Pour chaque exemple, affichez le label gold, le label prédit et le texte de l'exemple correspondant.      
 - Affichez également le score du modèle sur l'ensemble d'évaluation.

Note: aidez vous de la doc de Trainer https://huggingface.co/docs/transformers/main_classes/trainer#transformers.Trainer



In [27]:
#%pip -q install --upgrade transformers accelerate
%pip -q install accelerate==0.15.0 transformers==4.28.1

Note: you may need to restart the kernel to use updated packages.


In [42]:
import torch
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
device    

device(type='mps')

In [43]:

#print(f'small_eval_dataset :  {small_eval_dataset}')

err_predlabels=[]

if training_args.do_eval :
    prob_labels, _,_ , = trainer.predict(test_dataset=small_eval_dataset)
    print(f'prob_labels: {prob_labels}')
    pred_labels = [ np.argmax(logits, axis=-1) for logits in prob_labels]
    print(f'pred_labels: {pred_labels}')
    for i, inst in enumerate (small_eval_dataset):
        if pred_labels[i] != inst["label"] :
           err_predlabels.append((inst["label"], pred_labels[i],prob_labels[i] ))
           print(f' inst :  {inst["label"] } pred_labels[{i}]  {pred_labels[i]} : prob_labels[{i}] : {prob_labels[i]}')
           print("gold", small_eval_dataset[i]["label"])


#    for i in range ( len(small_eval_dataset)):
# ex_eval = small_eval_dataset[1]["text"]
# input = tokenizer(ex_eval, return_tensors="pt")
# input_ids = input.input_ids.to(device)
# print(input_ids.shape)
# output = model(input_ids)
        

prob_labels: [[ 4.468193   -4.0866537 ]
 [-4.4163823   4.259544  ]
 [-3.130955    2.9501593 ]
 [ 0.30607596 -0.31331158]
 [-4.1093516   4.0655303 ]
 [-3.911173    3.731034  ]
 [-4.591643    4.3551145 ]
 [-4.1176705   3.8949225 ]
 [-3.6940966   3.5233061 ]
 [-4.5370846   4.275413  ]]
pred_labels: [0, 1, 1, 0, 1, 1, 1, 1, 1, 1]
 inst :  1 pred_labels[0]  0 : prob_labels[0] : [ 4.468193  -4.0866537]
gold 1
 inst :  0 pred_labels[2]  1 : prob_labels[2] : [-3.130955   2.9501593]
gold 0
 inst :  1 pred_labels[3]  0 : prob_labels[3] : [ 0.30607596 -0.31331158]
gold 1
 inst :  0 pred_labels[4]  1 : prob_labels[4] : [-4.1093516  4.0655303]
gold 0
 inst :  0 pred_labels[7]  1 : prob_labels[7] : [-4.1176705  3.8949225]
gold 0
 inst :  0 pred_labels[8]  1 : prob_labels[8] : [-3.6940966  3.5233061]
gold 0


# 3 - Interprétabilité

Dans cette partie nous allons tester une méthode "d'attribution" qui observe certains valeurs du modèle pour repérer les parties importantes de l'input dans la décision du modèle.

Nous utiliserons le package *transformers_interpret*, qui est une surcouche de la librairie plus générale *captum*.

- Captum library: https://captum.ai/

## 3.1 Classification de phrases: sentiment

In [ ]:
# pour utiliser un modèle existant répertorié sur huggingface.co
model_name = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

### ▶▶ Exercice : Afficher les attributions

Utiliser le *cls_explainer* défini ci-dessous pour afficher les attributions pour chaque mot pour :
- un exemple correctement prédit
- un exemple correspondant à une erreur du modèle
Utilisez également la fonction de visualisation des attributions.

Aidez-vous de l'exemple sur cette page : https://pypi.org/project/transformers-interpret/

In [ ]:
cls_explainer = SequenceClassificationExplainer(
    model,
    tokenizer)

### ▶▶ Exercice : chercher les termes corrélés à chaque classe

- Appliquer le modèle appris sur l'éval de imdb
- Appliquer l'interprétation sur un ensemble d'instances (100 puis 1000) et relever les termes avec les attributions les plus fortes, dans un sens ou dans l'autre. Réduisez la taille des phrases des reviews à 30 tokens.
- Trouvez les éventuels biais du jeu de données
- Pour mieux contrôler l'influence d'un mot on peut aussi regarder la différence entre son rôle positif/négatif (certains mots sont parfois considérés comme influents tout le temps). Les plus intéressants sont ceux qui ont l'écart maximal entre les deux valeurs.



## 3.2 Classification de tokens : entités nommées

## ▶▶ Exercice : Explication de modèle de reconnaissance d'entités nommées

On définit ci-dessous un modèle de reconnaissance d'entités nommées.
Utilisez l'outil d'explicabilité pour une tâche de classification de token, et affichez les attributions pour un exemple.

In [ ]:
model_name = 'dslim/bert-base-NER'
model = AutoModelForTokenClassification.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)